# 🧬 DNABERT-2 + mCNN Pipeline for Transcription Factor Binding Site Prediction

This notebook implements a state-of-the-art deep learning model combining pre-trained `zhihan1996/DNABERT-2-117M` embeddings with a Multi-Scale Convolutional Neural Network (mCNN) to predict transcription factor binding sites (SP1, SP2, SP4, and Negative).

## Architecture Overview
1. **DNABERT-2 (Pre-trained Foundation Model)**: Tokenizes DNA sequences and outputs rich contextualized representations of shape `(batch_size, sequence_length, 768)`.
2. **Multi-Scale CNN (mCNN)**: Scans the embeddings using parallel 1D convolutions of kernel sizes `[3, 5, 7, 9]` to extract motif features at multiple widths, pools them to achieve translation invariance, and classifies them via fully connected layers.

## Memory Efficiency (On-the-Fly Tokenization)
To prevent **System Out-Of-Memory (OOM)** kernel crashes, we do **not** pre-extract and store all embeddings in RAM. Instead, we use a PyTorch `Dataset` that tokenizes the sequences on-the-fly and runs DNABERT-2's forward pass dynamically during training batches. This is extremely memory efficient.

### 1. Environment Setup and Installation
Install all required packages. Works on both Colab and Kaggle.

In [ ]:
# Install dependencies
!pip install -q transformers einops scikit-learn matplotlib seaborn safetensors huggingface_hub

# Detect environment: Colab vs Kaggle
import os, subprocess, sys

REPO_URL = "https://github.com/JustinYuanZe/SP1_TF_Biding_Project.git"
REPO_NAME = "SP1_TF_Biding_Project"

# Clone repo if it doesn't exist
if not os.path.isdir(REPO_NAME):
    print("Cloning GitHub repository...")
    subprocess.run(["git", "clone", REPO_URL], check=True)
else:
    print(f"Repository '{REPO_NAME}' already exists, pulling latest...")
    subprocess.run(["git", "-C", REPO_NAME, "pull"], check=True)

os.chdir(REPO_NAME)
print(os.getcwd())

# Add src to path
if 'src' not in sys.path:
    sys.path.insert(0, '.')

### 2. Imports and Device Setup
Import all required modules and detect GPU availability.

In [ ]:
import torch
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
from IPython.display import Image, display

# Device setup
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Import modules from src/
from src.dnabert_wrapper import DNABERTWrapper
from src.mcnn_model import MultiScaleCNN
from src.pipeline_utils import DNAPipelineDataset, DNABERT_mCNN
from src.train import train_model, evaluate_model, plot_curves

### 3. Loading Dataset Sequences
Load the processed final dataset from `data/processed/`.

In [ ]:
sp1_path = os.path.join("data", "processed", "sp1_positive_final.fasta")
sp2_path = os.path.join("data", "processed", "sp2_positive_final.fasta")
sp4_path = os.path.join("data", "processed", "sp4_positive_final.fasta")
neg_path = os.path.join("data", "processed", "negative_final.fasta")

def load_fasta(path):
    seqs = []
    with open(path, 'r') as f:
        for line in f:
            line = line.strip()
            if not line.startswith('>'):
                seqs.append(line.upper())
    return seqs

print("Loading datasets...")
seqs_sp1 = load_fasta(sp1_path)
seqs_sp2 = load_fasta(sp2_path)
seqs_sp4 = load_fasta(sp4_path)
seqs_neg = load_fasta(neg_path)

sequences = seqs_sp1 + seqs_sp2 + seqs_sp4 + seqs_neg
y = np.concatenate([
    np.zeros(len(seqs_sp1)),
    np.ones(len(seqs_sp2)),
    np.full(len(seqs_sp4), 2),
    np.full(len(seqs_neg), 3)
], axis=0)

print(f"Total sequences: {len(sequences)}")
print(f"Labels distribution: {np.bincount(y.astype(int))}")

### 4. Load DNABERT-2 Base Model
We instantiate the `DNABERTWrapper` which loads the DNABERT-2 tokenizer and foundation model weights using robust fallback strategies.

In [ ]:
# Instantiate wrapper
wrapper = DNABERTWrapper(device=device)

### 5. Stratified Split and PyTorch Dataloaders
We split the raw sequence strings and integer labels, then construct memory-efficient on-the-fly tokenizing datasets.

In [ ]:
# Train/Validation Split (80% train, 20% validation) on raw sequence lists
seq_train, seq_val, y_train, y_val = train_test_split(
    sequences, y, test_size=0.2, stratify=y, random_state=42
)

# Create PyTorch datasets (tokenizes on-the-fly in loader thread)
train_dataset = DNAPipelineDataset(seq_train, y_train, wrapper.tokenizer, max_length=105)
val_dataset = DNAPipelineDataset(seq_val, y_val, wrapper.tokenizer, max_length=105)

# Dataloaders
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False, num_workers=2)

print(f"Train dataloader batches: {len(train_loader)}")
print(f"Val dataloader batches:   {len(val_loader)}")

### 6. Training the Combined DNABERT-2 + mCNN Pipeline
We build the end-to-end `DNABERT_mCNN` model. By default, we keep DNABERT-2's representation weights frozen to optimize speed and resource usage, but it can also be fine-tuned.

In [ ]:
# 1. Initialize mCNN architecture head
mcnn = MultiScaleCNN(embedding_dim=768, branch_channels=128, num_classes=4, dropout_rate=0.5)

# 2. Combine with DNABERT-2 backbone (frozen by default to run quickly and save memory)
pipeline_model = DNABERT_mCNN(dnabert_model=wrapper.model, mcnn_model=mcnn, freeze_dnabert=True)

# 3. Train for 15 epochs
history = train_model(
    model=pipeline_model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=15,
    lr=0.001,
    device=device,
    output_dir='models'
)

### 7. Performance Analysis and Plots
Let's plot the training convergence curves and evaluate our model's performance on the validation set using confusion matrix heatmaps and ROC/PR curves.

In [ ]:
# 1. Plot and save training convergence curves
plot_curves(history, save_dir='figures')
display(Image('figures/mcnn_training_curves.png'))

In [ ]:
# 2. Load the best saved model state
best_model_path = os.path.join("models", "best_mcnn_model.pt")
pipeline_model.load_state_dict(torch.load(best_model_path, map_location=device))
print(f"Loaded best model weights from {best_model_path}")

# 3. Run detailed evaluation metrics (Classification Report, CM, ROC, PR Curves)
class_names = ['SP1', 'SP2', 'SP4', 'Negative']
evaluate_model(pipeline_model, val_loader, class_names, device=device, save_dir='figures')

# Display Confusion Matrix and ROC Curves inline
print("\n--- Visualizing Performance Plots ---")
print("Confusion Matrix:")
display(Image('figures/confusion_matrix.png'))
print("ROC Curves:")
display(Image('figures/roc_curves.png'))
print("Precision-Recall Curves:")
display(Image('figures/precision_recall_curves.png'))